In [ ]:
import re
import shutil
from pathlib import Path

#!/usr/bin/env python3

# Resolve base (works in notebook and script)
try:
    BASE = Path(__file__).resolve().parent
except NameError:
    BASE = Path.cwd()

RUNS_DIR = BASE  # this notebook sits in medMNIST/runs
CLEAN_MODELS_DIR = BASE / "clean_models"

# Regex to match run directory names
# Format: modeltype_224_timestamp_randaug[01]_numepochs100_bs128[_dropout0.X]
RUN_DIR_REGEX = re.compile(
    r"^(?P<modeltype>resnet18|vit_b_16)_224_(?P<timestamp>.+?)_randaug(?P<randaug>[01])_numepochs\d+_bs\d+(?:_dropout(?P<dropout>0\.\d+))?$"
)

# Regex to extract fold number from .pt filenames (e.g., prefix_0.pt, prefix_4.pt)
FOLD_FILE_REGEX = re.compile(r"_(\d)\_augmented.pt$")

def parse_run_dir(run_name: str):
    """Parse run directory name and extract components."""
    m = RUN_DIR_REGEX.match(run_name)
    if not m:
        return None
    return {
        'modeltype': m.group('modeltype'),
        'randaug': m.group('randaug'),
        'dropout': m.group('dropout') if m.group('dropout') else None
    }

def find_fold_models(run_dir: Path):
    """Find all .pt files and extract fold numbers (0-4) from their names."""
    fold_models = {}
    
    # Search for all .pt files in the run directory
    for pt_file in run_dir.glob("*.pt"):
        # Extract fold number from filename (e.g., model_0.pt -> 0)
        match = FOLD_FILE_REGEX.search(pt_file.name)
        if match:
            fold_num = int(match.group(1))
            if 0 <= fold_num <= 4:  # Only accept folds 0-4
                fold_models[fold_num] = pt_file
    
    return fold_models

def build_new_filename(dataset: str, modeltype: str, randaug: str, dropout: str, fold_num: int) -> str:
    """Build new filename following format: dataset_modeltype_224_randaug[01]_dropout[01]_[num_fold].pt"""
    parts = [dataset, modeltype, "224", f"randaug{randaug}"]
    
    if dropout:
        # Convert dropout 0.3 -> dropout03, 0.1 -> dropout01, etc.
        dropout_str = dropout.replace("0.", "0")
        parts.append(f"dropout{dropout_str}")
    
    parts.append(f"fold_{fold_num}")
    return "_".join(parts) + ".pt"

def copy_and_rename_models(dataset: str, run_dir: Path, target_dir: Path):
    """Copy fold models from run directory to target with new names."""
    parsed = parse_run_dir(run_dir.name)
    if not parsed:
        return 0
    
    fold_models = find_fold_models(run_dir)
    if not fold_models:
        return 0
    
    copied_count = 0
    for fold_num, source_path in fold_models.items():
        new_filename = build_new_filename(
            dataset=dataset,
            modeltype=parsed['modeltype'],
            randaug=parsed['randaug'],
            dropout=parsed['dropout'],
            fold_num=fold_num
        )
        
        dest_path = target_dir / new_filename
        
        try:
            shutil.copy2(source_path, dest_path)
            print(f"✓ Copied: {source_path.relative_to(RUNS_DIR)} -> {dest_path.name}")
            copied_count += 1
        except Exception as e:
            print(f"✗ WARN: Failed to copy {source_path}: {e}")
    
    return copied_count

def main():
    if not RUNS_DIR.exists():
        print(f"Runs directory not found: {RUNS_DIR}")
        return
    
    # Create clean_models directory
    CLEAN_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Target directory: {CLEAN_MODELS_DIR}\n")
    
    total_models = 0
    
    # Iterate through dataset subdirectories
    dataset_dirs = [d for d in RUNS_DIR.iterdir() if d.is_dir() and d.name != "clean_models"]
    
    for dataset_dir in sorted(dataset_dirs):
        dataset_name = dataset_dir.name
        print(f"\n{'='*60}")
        print(f"Processing dataset: {dataset_name}")
        print(f"{'='*60}")
        
        # Find all run directories matching pattern
        run_dirs = [d for d in dataset_dir.iterdir() if d.is_dir() and RUN_DIR_REGEX.match(d.name)]
        
        if not run_dirs:
            print(f"  No matching run directories found")
            continue
        
        print(f"  Found {len(run_dirs)} run directory(ies)")
        
        for run_dir in sorted(run_dirs):
            print(f"\n  Run: {run_dir.name}")
            count = copy_and_rename_models(dataset_name, run_dir, CLEAN_MODELS_DIR)
            total_models += count
            if count > 0:
                print(f"    → Copied {count} model(s)")
    
    print(f"\n{'='*60}")
    print(f"✓ Complete! Total models copied: {total_models}")
    print(f"✓ Models saved to: {CLEAN_MODELS_DIR}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()
